# CVDLens 학습 노트북 (Kaggle)

**사용 전 체크리스트**
1. 오른쪽 Session options → **GPU T4 x2 또는 P100** 선택
2. 오른쪽 Input → Add Input → **두 가지 데이터셋 추가**:
   - COCO 2017: `awsaf49/coco-2017-dataset` 검색 후 추가
   - 코드 zip: 본인이 업로드한 `cvdlens-code` 데이터셋 추가
3. 셀을 위에서부터 순서대로 실행

> **체크포인트는 `/kaggle/working/`에 저장되며 세션 종료 후에도 유지됩니다.**

## 0. GPU 확인

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'Python: {sys.version}')

## 1. 입력 데이터셋 경로 확인

In [ ]:
import os

# 추가된 데이터셋 목록 확인
for d in os.listdir('/kaggle/input/'):
    print(f'/kaggle/input/{d}/')
    for f in os.listdir(f'/kaggle/input/{d}/')[:5]:
        print(f'  {f}')

## 2. 코드 압축 해제

위 출력에서 zip 파일 경로 확인 후 `ZIP_PATH` 수정

In [ ]:
import zipfile, shutil, os

# 위 셀 출력에서 zip 파일 경로 확인 후 수정
ZIP_PATH = '/kaggle/input/cvdlens-code/model-training.zip'
DST      = '/kaggle/working/cvdlens'

assert os.path.exists(ZIP_PATH), f'zip 경로 확인 필요: {ZIP_PATH}\n위 셀 출력에서 실제 경로 확인하세요.'

if os.path.exists(DST):
    shutil.rmtree(DST)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/kaggle/working')

if os.path.exists('/kaggle/working/model-training') and not os.path.exists(DST):
    os.rename('/kaggle/working/model-training', DST)

print('압축 해제 완료')
print(os.listdir(DST))

## 3. Python 버전 호환 패치

In [ ]:
import sys

if sys.version_info < (3, 10):
    path = '/kaggle/working/cvdlens/data/dataset.py'
    code = open(path).read()
    if 'str | Path' in code:
        code = 'from typing import Union\n' + code
        code = code.replace('coco_dir: str | Path', 'coco_dir: Union[str, Path]')
        open(path, 'w').write(code)
        print(f'Python {sys.version_info.major}.{sys.version_info.minor} 호환 패치 완료')
else:
    print(f'Python {sys.version_info.major}.{sys.version_info.minor} — 패치 불필요')

## 4. 패키지 설치

In [ ]:
%%capture
!pip install \
    segmentation-models-pytorch>=0.3.3 \
    pytorch-lightning>=2.0 \
    torchmetrics>=1.0 \
    onnx>=1.14 \
    onnxruntime>=1.16 \
    onnxconverter-common>=1.13 \
    scikit-image>=0.21 \
    tqdm>=4.65

In [ ]:
import torch, pytorch_lightning as pl, segmentation_models_pytorch as smp
print(f'PyTorch   : {torch.__version__}')
print(f'Lightning : {pl.__version__}')
print(f'smp       : {smp.__version__}')
print(f'CUDA      : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU       : {torch.cuda.get_device_name(0)}')
    print(f'VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 5. COCO 데이터 경로 설정

Kaggle 데이터셋으로 추가했으므로 다운로드 불필요

In [ ]:
import os

# 위 셀 1번 출력에서 실제 COCO 경로 확인 후 수정
LOCAL_COCO_DIR = '/kaggle/input/coco-2017-dataset/coco2017'

# 경로 확인
train_path = f'{LOCAL_COCO_DIR}/train2017'
val_path   = f'{LOCAL_COCO_DIR}/val2017'

assert os.path.exists(train_path), f'train2017 경로 없음: {train_path}\n셀 1번 출력에서 실제 경로 확인하세요.'
assert os.path.exists(val_path),   f'val2017 경로 없음: {val_path}'

train_count = len(list(os.scandir(train_path)))
val_count   = len(list(os.scandir(val_path)))
print(f'train2017: {train_count:,}장')
print(f'val2017  : {val_count:,}장')
print('COCO 경로 확인 완료')

## 6. 학습 설정

In [ ]:
OUTPUT_DIR = '/kaggle/working/outputs'

# 재개할 체크포인트 (처음부터면 None)
RESUME_CKPT = None
# RESUME_CKPT = '/kaggle/input/cvdlens-checkpoint/last.ckpt'

BATCH_SIZE = 16
MAX_EPOCHS = 100
LR         = 1e-3
NUM_TRAIN  = 10000
NUM_VAL    = 2000
NUM_TEST   = 2000

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'체크포인트 재개 : {RESUME_CKPT or "처음부터"}')
print(f'배치 크기       : {BATCH_SIZE}')
print(f'최대 에폭       : {MAX_EPOCHS} (EarlyStopping patience=15)')

## 7. 학습 실행

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/cvdlens')

import torch
import pytorch_lightning as pl
from pathlib import Path
from data.dataset import build_dataloaders
from model import CVDLitModule

pl.seed_everything(42, workers=True)

train_loader, val_loader, _ = build_dataloaders(
    coco_dir=LOCAL_COCO_DIR,
    batch_size=BATCH_SIZE,
    num_workers=2,
    num_train=NUM_TRAIN,
    num_val=NUM_VAL,
    num_test=NUM_TEST,
)

if RESUME_CKPT:
    print(f'체크포인트에서 재개: {RESUME_CKPT}')
    module = CVDLitModule.load_from_checkpoint(RESUME_CKPT)
else:
    print('처음부터 학습 시작')
    module = CVDLitModule(lr=LR, l1_w=1.0, ssim_w=0.5, perc_w=0.1)

ckpt_dir = Path(OUTPUT_DIR) / 'checkpoints'

callbacks = [
    pl.callbacks.ModelCheckpoint(
        dirpath=str(ckpt_dir),
        filename='best-{epoch:03d}-{val_loss:.4f}',
        monitor='val_loss',
        save_top_k=3,
        save_last=True,
        mode='min',
    ),
    pl.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        mode='min',
        verbose=True,
    ),
    pl.callbacks.LearningRateMonitor(logging_interval='epoch'),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    callbacks=callbacks,
    logger=pl.loggers.TensorBoardLogger(
        save_dir=str(Path(OUTPUT_DIR) / 'logs'),
        name='cvdlens',
    ),
    log_every_n_steps=50,
    precision='16-mixed',
)

trainer.fit(module, train_loader, val_loader, ckpt_path=RESUME_CKPT)

print(f'\n학습 완료')
print(f'최적 체크포인트 : {trainer.checkpoint_callback.best_model_path}')
print(f'최적 val_loss   : {trainer.checkpoint_callback.best_model_score:.4f}')

## 8. 보정 결과 시각화

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from pathlib import Path

best_ckpt   = trainer.checkpoint_callback.best_model_path
module_eval = CVDLitModule.load_from_checkpoint(best_ckpt).eval()
to_tensor   = transforms.ToTensor()
CVD_TYPES   = {'Protanopia': 0.0, 'Deuteranopia': 0.5, 'Tritanopia': 1.0}
val_imgs    = sorted(Path(f'{LOCAL_COCO_DIR}/val2017').glob('*.jpg'))[:3]

fig, axes = plt.subplots(len(val_imgs), 4, figsize=(16, 4 * len(val_imgs)))
fig.suptitle('원본 / Protanopia / Deuteranopia / Tritanopia')

for row, img_path in enumerate(val_imgs):
    img = Image.open(img_path).convert('RGB').resize((256, 256))
    rgb = to_tensor(img).unsqueeze(0)
    axes[row][0].imshow(img)
    axes[row][0].set_title('원본')
    axes[row][0].axis('off')
    with torch.no_grad():
        for col, (name, val) in enumerate(CVD_TYPES.items(), start=1):
            inp = torch.cat([rgb, torch.full((1,1,256,256), val)], dim=1)
            out = module_eval(inp).squeeze(0).permute(1,2,0).numpy()
            axes[row][col].imshow(np.clip(out, 0, 1))
            axes[row][col].set_title(name)
            axes[row][col].axis('off')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sample_result.png', dpi=150)
plt.show()

## 9. ONNX 변환

결과물은 `/kaggle/working/` 에 저장되어 세션 종료 후에도 유지됩니다.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/cvdlens')
from export_onnx import export

best_ckpt = trainer.checkpoint_callback.best_model_path
onnx_dir  = f'{OUTPUT_DIR}/onnx'

export(ckpt_path=best_ckpt, output_dir=onnx_dir, fp16=True)
print('\nONNX 파일 위치:')
for f in os.listdir(onnx_dir):
    size = os.path.getsize(f'{onnx_dir}/{f}') / 1e6
    print(f'  {onnx_dir}/{f}  ({size:.1f} MB)')